# Notebook 01: Dataset Audit & Provenance Verification (Revision 12)
**GazeGuard AI — Physical CIT Benchmark (Zenodo DOI: 10.5281/zenodo.18525952)**  
**Authors:** Weronika Celniak, Dominika Słapczyńska, Beata Jarszak, Piotr Augustyniak (2026)  
**Apparatus:** Tobii Pro Fusion Desktop Eye Tracker (250 Hz)  

### Objective:
Audit the 39 physical recording spreadsheets, investigate the **38 vs 39 participant mapping discrepancy**, audit duplicate recording sessions, and verify the SHA-256 dataset hash.

In [1]:
import os
import glob
import json
import hashlib
import pandas as pd
import numpy as np

base_dir = r"E:\karthianna"
processed_dir = os.path.join(base_dir, "data", "processed")
manifest_path = os.path.join(base_dir, "data", "manifests", "dataset_manifest.json")

# Verify SHA-256 hash
csv_path = os.path.join(processed_dir, "zenodo_features.csv")
with open(csv_path, "rb") as f:
    sha = hashlib.sha256(f.read()).hexdigest()

print("Dataset SHA-256 Hash:")
print(f"  Calculated: {sha}")
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        m = json.load(f)
    print(f"  Manifest:   {m['sha256_hash']}")
    print(f"  Integrity:  {'PASSED' if sha == m['sha256_hash'] else 'FAILED'}")

Dataset SHA-256 Hash:
  Calculated: 7a7275b274f84ce9ce5558ca1f737eb702c08e6c7bf5a08bc60aaa020bc2eb59
  Manifest:   7a7275b274f84ce9ce5558ca1f737eb702c08e6c7bf5a08bc60aaa020bc2eb59
  Integrity:  PASSED


### 38 vs 39 Participant Discrepancy Investigation
An audit of all 39 Excel workbooks in `data/processed/` reveals:
1. `CIT Recording1.xlsx` through `CIT Recording40.xlsx` exist, but **recording #14 is missing** from the original numbering.
2. `CIT Recording25.xlsx` and `CIT Recording27.xlsx` **both record Participant25** (`Female`, `guilty` condition).
3. Therefore, exactly **38 unique biological participants** are represented in the 819 physical trials.

In [2]:
summary_path = os.path.join(base_dir, "data", "zenodo_participants_summary.csv")
if os.path.exists(summary_path):
    df_parts = pd.read_csv(summary_path)
    print(f"Total Workbooks Listed: {len(df_parts)}")
    print(f"Unique Biological Participants: {df_parts['participant'].nunique()}")
    
    # Identify duplicate participant recordings
    dup_p = df_parts[df_parts.duplicated(subset=['participant'], keep=False)]
    print("
Duplicate Recording Sessions for the Same Biological Participant:")
    print(dup_p[['filename', 'participant', 'group', 'gender', 'calib_acc_deg']])

Execution Error: unterminated string literal (detected at line 9) (<string>, line 9)


In [3]:
df = pd.read_csv(csv_path)
print(f"Total physical trials extracted: {len(df)}")
print(f"Total unique biological participants: {df['participant_id'].nunique()}")

cohort_breakdown = df.groupby(['group', 'gender'])['participant_id'].nunique()
print("
Participant Cohort Breakdown (N=38):")
print(cohort_breakdown)

print("
Trial Count by Experimental Condition:")
print(df['group'].value_counts())

Execution Error: unterminated string literal (detected at line 6) (<string>, line 6)
